# Superposition, proximité et mesures vectorielles

**Cartomize · Version 1.0**

Durée indicative : 35 minutes.

## Objectifs

Contrôler les géométries, combiner des secteurs, rattacher des stations et calculer des distances et des superficies.

## Apport de Cartomize

Les opérations retournent des GeoDataFrame compatibles avec GeoPandas et peuvent être réutilisées dans un plan automatique.

Données : paysage pédagogique généré avec une graine fixe. Les cartes et mesures ne décrivent aucun site réel.

## Préparation

Installer l’environnement décrit dans le [guide de démarrage](../README.md), puis exécuter les cellules dans l’ordre. Chaque exécution utilise un nouveau dossier de résultats.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
from IPython.display import display, Image
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'course.py').is_file() or (p/'tutorials/course.py').is_file())
root = root if (root/'course.py').is_file() else root/'tutorials'
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import cartomize as cm
from course import demo, run_directory, show_raster, show_rgb, gallery, CRS, CLASSES, ROLES

In [2]:
out = run_directory("01_superposition_vectorielle")
d = demo(out/"data")

## 1. Préparer les couches

Les distances sont exprimées en mètres dans un SCR projeté. Une réparation géométrique corrige la topologie ; elle ne confirme pas la position réelle des limites.

In [3]:
zones, points, roads = [cm.read_file(d[k]) for k in ('zones','points','roads')]
display(cm.validate(zones))
zones = cm.make_valid(zones)
latlon = cm.reproject(points, 'EPSG:4326')
coordinates = pd.DataFrame({'longitude': latlon.geometry.x, 'latitude': latlon.geometry.y})
rebuilt = cm.from_xy(coordinates)
assert len(rebuilt) == len(points)

{'features': 2,
 'crs': 'EPSG:32733',
 'missing_geometries': 0,
 'empty_geometries': 0,
 'invalid_geometries': 0,
 'duplicate_geometries': 0,
 'geometry_types': {'Polygon': 2}}

## 2. Créer des zones et des relations

Le tampon représente une distance géométrique autour des stations. Il ne constitue pas une preuve d’impact ni une zone d’accessibilité routière.

In [4]:
tampons = cm.buffer(points, 450, metric_crs=CRS)
intersections = cm.overlay(tampons, zones, how='intersection')
union = cm.overlay(tampons, zones, how='union')
difference = cm.overlay(zones, tampons, how='difference')
symetrique = cm.overlay(zones, tampons, how='symmetric_difference')
selection = cm.clip(roads, cm.read_file(d['aoi']))
affectation = cm.sjoin(points, zones, predicate='within')
proximite = cm.nearest(points, roads, metric_crs=CRS)
fusion = cm.dissolve(zones, by='group')
intersections['surface_ha'] = cm.area(intersections, metric_crs=CRS)
roads['longueur_m'] = cm.length(roads, metric_crs=CRS)
display(affectation.drop(columns='geometry'))
display(intersections.drop(columns='geometry'))

,name_left,index_right,name_right,group
0,Station A,0,Secteur ouest,Ensemble
1,Station B,0,Secteur ouest,Ensemble
2,Station C,1,Secteur est,Ensemble


,name_1,name_2,group,surface_ha
0,Station A,Secteur ouest,Ensemble,63.515107
1,Station B,Secteur ouest,Ensemble,33.555588
2,Station B,Secteur est,Ensemble,29.959519
3,Station C,Secteur est,Ensemble,63.515107


In [5]:
carte = cm.compose_map([
    {'data': zones, 'name':'Secteurs', 'color':'#e1e7e4'},
    {'data': tampons, 'name':'Distance de 450 m', 'color':'#b7d0bd', 'alpha':.65},
    {'data': roads, 'name':'Axes simulés', 'color':'#646464'},
    {'data': points, 'name':'Stations', 'labels':'name', 'color':'#202020'},
], title='Analyse de proximité', crs=CRS, credits='Données synthétiques | Distances planaires')
carte.export(out/'proximite.pdf', dpi=300)
fig = carte.render(dpi=130); gallery(fig,'01_proximite'); plt.show()

## 3. Vérifier les unités et conserver les résultats

In [6]:
assert np.isclose(cm.area(fusion).sum(), cm.area(zones).sum())
assert len(affectation) == len(points)
intersections.to_file(out/'intersections.gpkg', index=False)
roads.drop(columns='geometry').to_csv(out/'longueurs.csv', index=False)
display(cm.vector.analyze(zones))

{'features': 2,
 'crs': 'EPSG:32733',
 'missing_geometries': 0,
 'empty_geometries': 0,
 'invalid_geometries': 0,
 'duplicate_geometries': 0,
 'geometry_types': {'Polygon': 2},
 'name': '',
 'sampled_features': 2,
 'role': 'zones_thématiques',
 'role_confidence': 0.6,
 'label_field': 'name',
 'thematic_field': '',
 'fields': [{'name': 'name',
   'type_name': 'String',
   'count': 2,
   'null_count': 0,
   'null_percent': 0.0,
   'unique_count': 2,
   'unique_ratio': 1.0,
   'minimum': None,
   'maximum': None,
   'median': None,
   'mean': None,
   'skewness': None,
   'semantic_role': 'label',
   'recommended_use': 'Étiquetage',
   'confidence': 0.93},
  {'name': 'group',
   'type_name': 'String',
   'count': 2,
   'null_count': 0,
   'null_percent': 0.0,
   'unique_count': 1,
   'unique_ratio': 0.5,
   'minimum': None,
   'maximum': None,
   'median': None,
   'mean': None,
   'skewness': None,
   'semantic_role': 'category',
   'recommended_use': 'Catégories',
   'confidence': 0.74}

## Résultats de référence

![Résultat du cours](../gallery/01_proximite.png)

## Exercice

Comparer les surfaces des tampons à 250 m et 450 m. Pourquoi ne faut-il pas sommer directement leurs superficies s’ils se recouvrent ?

<details>
<summary>Éléments de correction</summary>

Les recouvrements seraient comptés plusieurs fois. Dissoudre les tampons avant de mesurer leur emprise cumulée : `cm.buffer(points, 450, dissolve=True, metric_crs=CRS)`.

</details>

## Reproduction

Les paramètres, la graine et les chemins sont explicites dans les cellules. Le répertoire `out` contient les produits de cette exécution. Adapter les sources, le système de coordonnées et les paramètres avant de transférer la méthode à une étude.